# Building a netlist programmatically

This notebook builds a heated Mach-Zehnder interferometer (MZI) netlist from
plain Python data with `kfnetlist`, then checks, flattens, and serializes it.

1. A reusable `arm` cell (heater + waveguide)
2. A top-level `mzi` cell built from a connection table, with arm instances
   that point to the `arm` netlist through `netlist_id`
3. An instance array (a pad row) wired with `PortArrayRef`
4. Validation (`validate_hierarchy`, `detect_opens`)
5. Flattening, a JSON round trip, and an LVS-style net diff

In [1]:
import json
from typing import Any

from kfnetlist import (
    Netlist,
    NetlistPort,
    PortArrayRef,
    PortRef,
    RefNetlistInstance,
    hierarchy_from_json,
    validate_hierarchy,
)

PDK = "demo_pdk"


def label(m: NetlistPort | PortRef) -> str:
    """`<port>` for cell ports, `inst.port` / `inst[ia,ib].port` for instance ports."""
    if isinstance(m, PortArrayRef):
        return f"{m.instance}[{m.ia},{m.ib}].{m.port}"
    if isinstance(m, PortRef):
        return f"{m.instance}.{m.port}"
    return f"<{m.name}>"


def show(nl: Netlist) -> None:
    """Print one line per net: `name: member, member, ...`."""
    for net in nl.nets:
        print(f"  {net.name or '(unnamed)':>10}: {', '.join(map(label, net))}")

## 1. A leaf-level cell: `arm`

One arm is a straight waveguide with a thermal phase shifter on it. It exposes
two optical ports (`o1`, `o2`) and two electrical ports (`h1`, `h2`).

In [2]:
arm = Netlist()
arm.create_inst("wg", kcl=PDK, component="straight", settings={"length": 100.0})
arm.create_inst("ht", kcl=PDK, component="heater", settings={"length": 100.0})

for port, inst, inst_port in [
    ("o1", "wg", "o1"),
    ("o2", "wg", "o2"),
    ("h1", "ht", "e1"),
    ("h2", "ht", "e2"),
]:
    arm.create_net(arm.create_port(port), PortRef(inst, inst_port), name=port)

show(arm)

          o1: <o1>, wg.o1
          o2: <o2>, wg.o2
          h1: <h1>, ht.e1
          h2: <h2>, ht.e2


## 2. The top cell: `mzi`

Instances and connections come from plain Python data, the way you would
write them when generating netlists from a PDK, a YAML file, or a script.

The two arms use `netlist_id="arm"`, so they become `RefNetlistInstance`s:
they point to the `arm` netlist in the same document. Instances without
`netlist_id` are leaf components (`LeafNetlistInstance`).

In [3]:
INSTANCES: dict[str, dict[str, Any]] = {
    "splitter": dict(component="mmi1x2"),
    "combiner": dict(component="mmi1x2"),
    "top": dict(component="arm", netlist_id="arm"),
    "bot": dict(component="arm", netlist_id="arm"),
}

# (net name, [members]); "inst.port" strings for instance ports, "<name>" for cell ports
CONNECTIONS = [
    ("in", ["<in>", "splitter.o1"]),
    ("split_top", ["splitter.o2", "top.o1"]),
    ("split_bot", ["splitter.o3", "bot.o1"]),
    ("comb_top", ["top.o2", "combiner.o2"]),
    ("comb_bot", ["bot.o2", "combiner.o3"]),
    ("out", ["combiner.o1", "<out>"]),
    ("GND", ["top.h2", "bot.h2", "<gnd>"]),
]


def member(token: str) -> NetlistPort | PortRef:
    if token.startswith("<"):
        return NetlistPort(token.strip("<>"))
    inst, port = token.split(".")
    return PortRef(inst, port)


mzi = Netlist()
for name, spec in INSTANCES.items():
    mzi.create_inst(name, kcl=PDK, **spec)
for port in ("in", "out", "gnd"):
    mzi.create_port(port)
for net_name, tokens in CONNECTIONS:
    mzi.create_net(*(member(t) for t in tokens), name=net_name)

for name, inst in mzi.instances.items():
    print(f"{name:>9}: {type(inst).__name__:<20} component={inst.component}")

 splitter: LeafNetlistInstance  component=mmi1x2
 combiner: LeafNetlistInstance  component=mmi1x2
      top: RefNetlistInstance   component=arm
      bot: RefNetlistInstance   component=arm


## 3. Instance arrays and `PortArrayRef`

`na` / `nb` create an instance array. A `PortArrayRef(instance, port, ia, ib)`
points to one element of it. Here a 2x1 pad row drives the two heaters.

In [4]:
mzi.create_inst("pads", kcl=PDK, component="pad", settings={"size": 80.0}, na=2, nb=1)

for ia, arm_name in enumerate(("top", "bot")):
    mzi.create_net(
        PortArrayRef("pads", "e1", ia, 0),
        PortRef(arm_name, "h1"),
        name=f"heater_{arm_name}",
    )

mzi.sort()
show(mzi)

         GND: <gnd>, bot.h2, top.h2
          in: <in>, splitter.o1
         out: <out>, combiner.o1
  heater_bot: bot.h1, pads[1,0].e1
   split_bot: bot.o1, splitter.o3
    comb_bot: bot.o2, combiner.o3
    comb_top: combiner.o2, top.o2
   split_top: splitter.o2, top.o1
  heater_top: top.h1, pads[0,0].e1


## 4. Validate

`validate_hierarchy` checks every `netlist_id` against the document and
rejects dangling references and cycles. `detect_opens` reports top-level ports
with no net and nets with a single member.

In [5]:
document = {"mzi": mzi, "arm": arm}
validate_hierarchy(document)
print("hierarchy ok")
print("opens:", mzi.detect_opens())

hierarchy ok
opens: {'unconnected_ports': [], 'singleton_nets': []}


In [6]:
# A dangling reference is rejected
try:
    validate_hierarchy({"mzi": mzi})
except ValueError as err:
    print("ValueError:", err)

ValueError: netlist "mzi", instance "bot": netlist_id "arm" does not exist in this document


## 5. Flatten

`flatten` replaces each referenced arm with its contents. Instance names get
the parent prefix, and nets that passed through an arm port are merged.

In [7]:
flat = mzi.flatten(document, recursive=True)
flat.sort()
print("instances:", flat.instance_names())
show(flat)

instances: ['bot.ht', 'bot.wg', 'combiner', 'pads', 'splitter', 'top.ht', 'top.wg']
         GND: <gnd>, bot.ht.e2, top.ht.e2
          in: <in>, splitter.o1
         out: <out>, combiner.o1
  heater_bot: bot.ht.e1, pads[1,0].e1
   split_bot: bot.wg.o1, splitter.o3
    comb_bot: bot.wg.o2, combiner.o3
    comb_top: combiner.o2, top.wg.o2
   split_top: splitter.o2, top.wg.o1
  heater_top: top.ht.e1, pads[0,0].e1


## 6. Serialize

`to_json` / `from_json` round-trip a single netlist. A whole document is a
JSON object of netlists, and `hierarchy_from_json` loads and validates it in
one step. Named nets are written as `{"name": ..., "members": [...]}`.

In [8]:
doc_json = json.dumps({name: json.loads(nl.to_json()) for name, nl in document.items()})
restored = hierarchy_from_json(doc_json)
assert restored["mzi"] == mzi and restored["arm"] == arm
assert isinstance(restored["mzi"].get_instance("top"), RefNetlistInstance)

print(json.dumps(json.loads(arm.to_json())["nets"][0], indent=2))

{
  "name": "o1",
  "members": [
    {
      "name": "o1"
    },
    {
      "instance": "wg",
      "port": "o1"
    }
  ]
}


## 7. Compare against a reference (LVS-style)

Say the layout has the bottom arm's ground strap missing. `find_net_difference`
compares nets by membership, so the names do not need to match.

In [9]:
data = mzi.to_dict()
for net in data["nets"]:
    if isinstance(net, dict) and net.get("name") == "GND":
        net["members"].remove({"instance": "bot", "port": "h2"})
layout = Netlist.from_dict(data)

diff = layout.find_net_difference(mzi)
for kind in ("missing", "extra"):
    print(f"{kind}:")
    for net in diff[kind]:
        print("   ", ", ".join(map(label, net)))

missing:
    <gnd>, bot.h2, top.h2
extra:
    <gnd>, top.h2
